# Scraper de datos musicales
Scraping de letras (AZLyrics) y géneros (Last.fm) a partir de un CSV propio.


In [5]:
# Ejecuta esta celda solo la primera vez
%pip install cloudscraper beautifulsoup4 pandas requests

## Imports y configuración

In [2]:
import re
import time
import random
import logging
import pandas as pd
import cloudscraper
import requests
from bs4 import BeautifulSoup, Comment
from urllib.parse import quote

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

# Cloudscraper (bypassea Cloudflare en AZLyrics)
SCRAPER = cloudscraper.create_scraper(
    browser={"browser": "chrome", "platform": "windows", "mobile": False}
)

# Cabeceras Last.fm 
HEADERS_LASTFM = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
}

# Delays entre requests
AZLYRICS_DELAY = (5, 10)   # AZLyrics tiene rate-limiting agresivo
LASTFM_DELAY   = (1, 3)

## Funciones de scraping

In [7]:
# "The Beatles" -> "thebeatles" para usarlo en la URL
def _slug(text: str) -> str:
    return re.sub(r"[^a-z0-9]", "", text.lower())

def _get_lastfm(url: str):
    time.sleep(random.uniform(*LASTFM_DELAY))
    try:
        resp = SCRAPER.get(url, headers=HEADERS_LASTFM, timeout=15)
        resp.raise_for_status()
        return resp
    except Exception as e:
        log.warning(f"Last.fm request failed: {url} → {e}")
        return None
    
def _get_azlyrics(url: str):
    time.sleep(random.uniform(*AZLYRICS_DELAY))
    try:
        resp = SCRAPER.get(url, timeout=20)
        resp.raise_for_status()
        return resp
    except Exception as e:
        log.warning(f"AZLyrics request failed: {url} → {e}")
        return None

def scrape_lyrics(artist: str, song: str):
    url = f"https://www.azlyrics.com/lyrics/{_slug(artist)}/{_slug(song)}.html"
    log.info(f"AZLyrics → {url}")
    resp = _get_azlyrics(url)
    if resp is None:
        return None
    if "ringtone" not in resp.text:
        log.warning(f"Página sin div.ringtone (canción no indexada)")
        return None
    soup = BeautifulSoup(resp.text, "html.parser")
    # Método 1: hermano de div.ringtone
    ringtone = soup.find("div", class_="ringtone")
    if ringtone:
        div = ringtone.find_next_sibling("div")
        if div:
            text = div.get_text(separator="\n").strip()
            if len(text) > 80:
                log.info(f"Letra encontrada ({len(text)} chars)")
                return text
    # Método 2: comentario HTML
    for comment in soup.find_all(string=lambda t: isinstance(t, Comment) and "Usage of azlyrics" in t):
        div = comment.find_next("div")
        if div:
            text = div.get_text(separator="\n").strip()
            if len(text) > 80:
                return text
    # Método 3: div sin class largo
    for div in soup.find_all("div", class_=False, id=False):
        text = div.get_text(separator="\n").strip()
        if len(text) > 150 and text.count("\n") > 5:
            if not any(s in text[:60] for s in ["AZLyrics", "Submit", "Login", "©"]):
                return text
    log.warning(f"No se encontró letra para {artist} - {song}")
    return None

def _extract_lastfm_tags(html: str, top_n: int):
    soup = BeautifulSoup(html, "html.parser")
    tags = []
    for selector in [".catalogue-tags .tag", ".catalogue-tags a",
                     ".tag-list .tag", ".tags-list a", "ul.tags li a", "a.tag"]:
        for el in soup.select(selector):
            tag = el.get_text(strip=True).lower()
            if tag and tag not in tags and len(tag) > 1:
                tags.append(tag)
        if tags:
            break
    if not tags:
        seen = set()
        for a in soup.find_all("a", href=True):
            if "/tag/" in a["href"]:
                tag = a.get_text(strip=True).lower()
                if tag and tag not in seen and len(tag) > 1:
                    tags.append(tag); seen.add(tag)
    return tags[:top_n]

def scrape_genre(artist: str, song: str = None, top_n: int = 3):
    tags = []
    if song:
        resp = _get_lastfm(f"https://www.last.fm/music/{quote(artist)}/_/{quote(song)}")
        if resp:
            tags = _extract_lastfm_tags(resp.text, top_n)
    if not tags:
        resp = _get_lastfm(f"https://www.last.fm/music/{quote(artist)}")
        if resp:
            tags = _extract_lastfm_tags(resp.text, top_n)
    log.info(f"Géneros: {tags}")
    return tags

def scrape_dataset(songs: list, output_csv: str = "music_dataset.csv") -> pd.DataFrame:
    results = []
    for i, entry in enumerate(songs, 1):
        artist, song = entry["artist"], entry["song"]
        log.info(f"[{i}/{len(songs)}] {artist} — {song}")
        lyrics = scrape_lyrics(artist, song)
        genres = scrape_genre(artist, song, top_n=3)
        results.append({
            "artist":  artist, "song": song, "lyrics": lyrics,
            "genres":  ", ".join(genres) if genres else None,
            "genre_1": genres[0] if len(genres) > 0 else None,
            "genre_2": genres[1] if len(genres) > 1 else None,
            "genre_3": genres[2] if len(genres) > 2 else None,
        })
        pd.DataFrame(results).to_csv(output_csv, index=False, encoding="utf-8")
    log.info(f"Dataset guardado: {output_csv} ({len(results)} canciones)")
    return pd.DataFrame(results)

print("Funciones cargadas")


Funciones cargadas


## Cargar y unir CSVs

In [8]:
import json
from pathlib import Path

# ── Credenciales Kaggle ──────────────────────────────────────────────────────
KAGGLE_USERNAME = ""   # tu usuario de Kaggle
KAGGLE_KEY      = ""   # tu API key de Kaggle

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
(kaggle_dir / "kaggle.json").write_text(json.dumps({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}))
(kaggle_dir / "kaggle.json").chmod(0o600)

# ── Descarga del dataset ─────────────────────────────────────────────────────
%pip install kaggle -q  # descomenta si no tienes kaggle instalado
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi(); api.authenticate()
api.dataset_download_files("neisse/scrapped-lyrics-from-6-genres", path=".", unzip=True, quiet=False)

songs_df   = pd.read_csv("lyrics-data.csv")
artists_df = pd.read_csv("artists-data.csv")

print("lyrics-data.csv columnas:", songs_df.columns.tolist())
print("artists-data.csv columnas:", artists_df.columns.tolist())

# Join por ALink (songs) <-> Link (artists)
merged = songs_df.merge(artists_df[["Link", "Artist"]], left_on="ALink", right_on="Link", how="left")

# Filtrar inglés y construir lista
all_songs = [
    {"artist": row["Artist"].strip(), "song": row["SName"].strip()}
    for _, row in merged.iterrows()
    if pd.notna(row["SName"]) and pd.notna(row["Artist"])
    and row.get("language") == "en"
]

# Rango de canciones a scrapear (índices basados en 1, ambos incluidos)
# Cambia START y END según el tramo que quieras procesar
START = 20000       # primera canción a scrapear
END   = 30000   # última canción a scrapear (None = hasta el final)

songs = all_songs[START - 1 : END][::100]

print(f"Total canciones en inglés: {len(all_songs)}")
print(f"Rango seleccionado:        {START} → {END or len(all_songs)}")
print(f"Canciones a scrapear:      {len(songs)} (cada 100)")
songs[:3]  # preview

ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


Note: you may need to restart the kernel to use updated packages.
Dataset URL: https://www.kaggle.com/datasets/neisse/scrapped-lyrics-from-6-genres


KeyboardInterrupt: 

## Ejecutar scraping
Con 1000 canciones este proceso tarda **3-5 horas**. El CSV se guarda tras cada canción,
así que puedes interrumpirlo y usar la celda de *Reanudar* para continuar.


In [ ]:
OUTPUT_CSV = f"music_dataset_{START}_{END}.csv"

df = scrape_dataset(songs, output_csv=OUTPUT_CSV)

print(f"Total canciones: {len(df)}")
print(f"Con letra:       {df['lyrics'].notna().sum()}")
print(f"Con géneros:     {df['genres'].notna().sum()}")


2026-05-02 21:15:51,131 [INFO] [1/101] Tarja Turunen — Until Silence
2026-05-02 21:15:51,133 [INFO] AZLyrics → https://www.azlyrics.com/lyrics/tarjaturunen/untilsilence.html
2026-05-02 21:15:58,533 [WARNING] AZLyrics request failed: https://www.azlyrics.com/lyrics/tarjaturunen/untilsilence.html → 404 Client Error: Not Found for url: https://www.azlyrics.com/lyrics/tarjaturunen/untilsilence.html
2026-05-02 21:16:01,874 [WARNING] Last.fm request failed: https://www.last.fm/music/Tarja%20Turunen/_/Until%20Silence → 502 Server Error: Bad Gateway for url: https://www.last.fm/music/Tarja%20Turunen/_/Until%20Silence
2026-05-02 21:16:04,847 [INFO] Géneros: ['symphonic metal', 'female vocalists', 'female fronted metal']
2026-05-02 21:16:04,852 [INFO] [2/101] Susan Boyle — You Have To Be There
2026-05-02 21:16:04,853 [INFO] AZLyrics → https://www.azlyrics.com/lyrics/susanboyle/youhavetobethere.html
2026-05-02 21:16:10,515 [INFO] Letra encontrada (1520 chars)
2026-05-02 21:16:13,316 [INFO] Género

KeyboardInterrupt: 

## Reanudar scraping interrumpido

In [ ]:
import os

OUTPUT_CSV = "music_dataset.csv"

if os.path.exists(OUTPUT_CSV):
    done_df   = pd.read_csv(OUTPUT_CSV)
    done_keys = set(zip(done_df["artist"].str.lower(), done_df["song"].str.lower()))
    pending   = [s for s in songs if (s["artist"].lower(), s["song"].lower()) not in done_keys]
    print(f"Ya procesadas: {len(done_df)} | Pendientes: {len(pending)}")

    if pending:
        new_df   = scrape_dataset(pending, output_csv="music_dataset_new.csv")
        combined = pd.concat([done_df, new_df], ignore_index=True)
        combined.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
        print(f"Dataset combinado: {len(combined)} canciones")
    else:
        print("Todas las canciones ya están procesadas")
else:
    print("No existe music_dataset.csv, ejecuta primero la celda de scraping")


## Explorar resultados

In [ ]:
df = pd.read_csv("music_dataset.csv")

print(f"Total canciones:  {len(df)}")
print(f"Con letra:        {df['lyrics'].notna().sum()} ({df['lyrics'].notna().mean():.0%})")
print(f"Con géneros:      {df['genres'].notna().sum()} ({df['genres'].notna().mean():.0%})")
print(f"\nGéneros más frecuentes:")
print(df["genre_1"].value_counts().head(10).to_string())


Total canciones:  101
Con letra:        66 (65%)
Con géneros:      71 (70%)

Géneros más frecuentes:
genre_1
country            44
pop                 4
bluegrass           4
rockabilly          2
oldies              2
60s                 2
symphonic metal     1
electropop          1
christmas           1
perquien            1


In [ ]:
# Vista previa de las primeras filas
df[["artist", "song", "genres", "lyrics"]].head(10)

,artist,song,genres,lyrics
0,Tarja Turunen,Until Silence,"symphonic metal, female vocalists, female fron...",NaN
1,Susan Boyle,You Have To Be There,"pop, love it, female vocalists",What is it Lord that you want\n\nThat I am not...
2,Taylor Swift,Cruel Summer,"electropop, pop, synthpop","(Yeah, yeah, yeah, yeah)\n\n\n\nFever dream hi..."
3,Taylor Swift,London Boy,NaN,[Idris Elba and James Corden:]\n\nWe can go dr...
4,Taylor Swift,Sugar,NaN,What a thing to see\n\nWhat a thing to be\n\nW...
5,Shania Twain,Forever And For Always,"country, pop, female vocalists","Mm, mm\n\nMm, in your arms\n\nOh\n\nI can hear..."
6,Shania Twain,Wild and Wicked,NaN,Full moon cravin' just a little\n\nYoung love ...
7,Johnny Cash,A Croft In Clachan,NaN,NaN
8,Johnny Cash,Devil's Right Hand,"country, folk, singer-songwriter",NaN
9,Johnny Cash,I Heard The Bells On Christmas Day,"christmas, country, folk",I heard the bells on Christmas Day\n\nTheir ol...


## Enriquecer dataset con años de publicación (Wikipedia)
Esta función busca el año de publicación de cada canción en Wikipedia.
Lee `music_dataset.csv`, añade la columna `year` y guarda el resultado
en el mismo archivo (o en uno nuevo si se especifica).

In [16]:
import re
import time
import random
import logging
import pandas as pd
import wikipediaapi

log = logging.getLogger(__name__)

WIKI_DELAY = (1, 3)

# Palabras de lanzamiento — ampliadas
_RELEASE_RE = re.compile(
    r"\b(released?|published|issued|single|debut|came out|"
    r"launch|record|studio album|EP|B-side|track|discograph)\b",
    re.IGNORECASE,
)
# Palabras biográficas — solo para descartar si NO hay keyword de release
_BIO_RE = re.compile(
    r"\b(born|birth|founded|formed|died|death)\b",
    re.IGNORECASE,
)
# Palabras que indican que la frase habla de la canción directamente
_SONG_CONTEXT_RE = re.compile(
    r"\b(song|single|track|record|album|music|written|composed|"
    r"produced|performed|band|group|artist)\b",
    re.IGNORECASE,
)

_wiki = wikipediaapi.Wikipedia(
    user_agent="MusicDatasetResearch/1.0 (educational project)",
    language="en",
    extract_format=wikipediaapi.ExtractFormat.WIKI,
)


def _clean_artist(raw: str) -> str:
    return raw.strip("/").strip()


def _title_case(text: str) -> str:
    # Respeta artículos y preposiciones cortas en minúscula (estilo Wikipedia)
    LOWER = {"a", "an", "the", "and", "but", "or", "in", "on", "at", "to", "of", "for"}
    words = text.split()
    return " ".join(
        w.lower() if (i > 0 and w.lower() in LOWER) else w.capitalize()
        for i, w in enumerate(words)
    )


def _get_page(title: str):
    time.sleep(random.uniform(*WIKI_DELAY))
    page = _wiki.page(title)
    return page if page.exists() else None


def _year_from_summary(summary: str) -> int | None:
    """
    Extrae el año de publicación del summary con tres niveles de confianza:
      1. Frase con keyword de release y sin keyword biográfica  (más fiable)
      2. Frase con contexto de canción/música y sin keyword bio  (fiable)
      3. Primera frase que contenga un año (fallback, solo si la página
         ya fue validada como sobre la canción por _page_is_about_song)
    """
    sentences = re.split(r"(?<=[.!?])\s+", summary)
    year_range = r"\b(19[5-9]\d|20[0-2]\d)\b"

    # Nivel 1 — release keyword explícita
    for s in sentences:
        if _RELEASE_RE.search(s) and not _BIO_RE.search(s):
            m = re.search(year_range, s)
            if m:
                return int(m.group(1))

    # Nivel 2 — contexto musical sin bio
    for s in sentences:
        if _SONG_CONTEXT_RE.search(s) and not _BIO_RE.search(s):
            m = re.search(year_range, s)
            if m:
                return int(m.group(1))

    # Nivel 3 — primer año que no sea bio pura
    for s in sentences:
        if _BIO_RE.search(s):
            continue
        m = re.search(year_range, s)
        if m:
            return int(m.group(1))

    return None


def _page_is_about_song(page, song: str) -> bool:
    """
    Comprueba que el artículo es sobre la canción y no sobre el artista.
    Estrategia en cascada: categorías → summary → título de página.
    """
    song_lower = song.lower()

    # 1. Categorías (más fiable)
    cats = " ".join(page.categories.keys()).lower()
    if cats:
        if re.search(r"\b(song|single|track)\b", cats):
            return True
        # Categorías presentes pero ninguna de canción: probablemente artista/álbum
        if song_lower not in page.summary[:600].lower():
            return False

    # 2. Summary menciona la canción
    if song_lower in page.summary[:600].lower():
        return True

    # 3. El título de la página contiene el nombre de la canción
    if song_lower in page.title.lower():
        return True

    return False


def scrape_year(artist_raw: str, song: str) -> int | None:
    """
    Devuelve el año de publicación de una canción buscando en Wikipedia.
    Acepta el artista en cualquier formato del dataset ('/moonspell/', etc.)
    """
    artist    = _clean_artist(artist_raw)
    artist_tc = _title_case(artist)
    song_tc   = _title_case(song)

    candidates = [
        f"{song_tc} ({artist_tc} song)",   # "Heartshaped Abyss (Moonspell song)"
        f"{song_tc} ({artist_tc})",         # "Heartshaped Abyss (Moonspell)"
        f"{song_tc} (song)",                # "Heartshaped Abyss (song)"
        song_tc,                            # "Heartshaped Abyss"
        f"{song} ({artist} song)",          # capitalización original
    ]
    seen = set()
    candidates = [c for c in candidates if not (c in seen or seen.add(c))]

    page = None
    for candidate in candidates:
        log.info(f"Probando: '{candidate}'")
        p = _get_page(candidate)
        if p and _page_is_about_song(p, song):
            page = p
            break

    if page is None:
        log.warning(f"No se encontró página para '{artist} - {song}'")
        return None

    log.info(f"Wikipedia → {page.fullurl}")
    year = _year_from_summary(page.summary)
    log.info(f"Año: {year}" if year else "Año no encontrado")
    return year


def enrich_with_years(
    input_csv: str = "music_dataset.csv",
    output_csv: str | None = None,
    overwrite_existing: bool = False,
    start: int | None = None,
    end: int | None = None,
) -> pd.DataFrame:
    """
    Añade la columna 'year' a un rango de filas del dataset.

    Parámetros
    ----------
    input_csv          : CSV de entrada.
    output_csv         : CSV de salida; None → nombre automático con rango.
    overwrite_existing : si False, omite filas que ya tienen 'year'.
    start              : primera fila (0-based, incluida). None = inicio.
    end                : última fila (0-based, excluida). None = final.

    Ejemplo para 4 personas (dataset de 1000 filas):
        enrich_with_years(start=0,   end=250)   → years_0_250.csv
        enrich_with_years(start=250, end=500)   → years_250_500.csv
        enrich_with_years(start=500, end=750)   → years_500_750.csv
        enrich_with_years(start=750, end=None)  → years_750_end.csv
    """
    df_full = pd.read_csv(input_csv)
    total   = len(df_full)

    row_start = start if start is not None else 0
    row_end   = end   if end   is not None else total

    if not (0 <= row_start < total):
        raise ValueError(f"start={row_start} fuera de rango (0–{total-1})")
    if not (row_start < row_end <= total):
        raise ValueError(f"end={row_end} fuera de rango ({row_start+1}–{total})")

    df = df_full.iloc[row_start:row_end].copy()

    if "year" not in df.columns:
        df["year"] = None

    if output_csv is None:
        end_label  = row_end if end is not None else "end"
        output_csv = f"years_{row_start}_{end_label}.csv"

    has_artist_clean = "artist_clean" in df.columns
    has_song_norm    = "song_norm"    in df.columns

    pending_mask = (
        df["year"].isna() if not overwrite_existing
        else pd.Series([True] * len(df), index=df.index)
    )
    pending_idx = df.index[pending_mask].tolist()

    log.info(
        f"Rango: filas {row_start}–{row_end-1} ({len(df)} canciones) | "
        f"Pendientes: {len(pending_idx)} | Salida: {output_csv}"
    )

    for count, idx in enumerate(pending_idx, 1):
        artist_raw   = df.at[idx, "artist_clean"] if has_artist_clean else df.at[idx, "artist"]
        song         = df.at[idx, "song_norm"]    if has_song_norm    else df.at[idx, "song"]
        song_display = df.at[idx, "song"]

        log.info(f"[{count}/{len(pending_idx)}] {artist_raw} — {song_display}")
        df.at[idx, "year"] = scrape_year(artist_raw, song)

        if count % 10 == 0 or count == len(pending_idx):
            df.to_csv(output_csv, index=False, encoding="utf-8")
            log.info(f"Guardado → {output_csv} ({count}/{len(pending_idx)})")

    log.info(f"Tramo completado. Con año: {df['year'].notna().sum()}/{len(df)}")
    return df


def merge_year_chunks(
    *chunk_csvs: str,
    base_csv: str = "music_dataset.csv",
    output_csv: str = "music_dataset_with_years.csv",
) -> pd.DataFrame:
    """
    Combina los CSV parciales de cada persona en el dataset completo.

    Uso:
        merge_year_chunks(
            "years_0_500.csv",
            "years_500_end.csv",
            base_csv="music_dataset.csv",
            output_csv="music_dataset_with_years.csv",
        )
    """
    base = pd.read_csv(base_csv)
    if "year" not in base.columns:
        base["year"] = None

    for path in chunk_csvs:
        chunk = pd.read_csv(path)[["artist", "song", "year"]].rename(columns={"year": "_yr"})
        base  = base.merge(chunk, on=["artist", "song"], how="left")
        base["year"] = base["year"].combine_first(base["_yr"])
        base.drop(columns=["_yr"], inplace=True)

    base.to_csv(output_csv, index=False, encoding="utf-8")
    log.info(f"Merge completado → {output_csv} | Con año: {base['year'].notna().sum()}/{len(base)}")
    return base


print("Funciones cargadas ✓")
print("  → scrape_year(artist_raw, song)")
print("  → enrich_with_years(input_csv, output_csv, overwrite_existing, start, end)")
print("  → merge_year_chunks(*csvs, base_csv, output_csv)")

2026-05-09 10:48:36,228 [INFO] Wikipedia: language=en, user_agent: MusicDatasetResearch/1.0 (educational project) (Wikipedia-API/0.14.1; https://github.com/martin-majlis/Wikipedia-API/), extract_format=1


Funciones cargadas ✓
  → scrape_year(artist_raw, song)
  → enrich_with_years(input_csv, output_csv, overwrite_existing, start, end)
  → merge_year_chunks(*csvs, base_csv, output_csv)


In [22]:
# ── Configuración del tramo ───────────────────────────────────────────────────
# Divide el dataset entre tantas personas como quieras.
# Ejemplo para 2 personas con un dataset de 1000 filas:
#   Persona 1 → START=0,   END=500
#   Persona 2 → START=500, END=None  (None = hasta el final)
#
# Ejemplo para 4 personas:
#   Persona 1 → START=0,   END=250
#   Persona 2 → START=250, END=500
#   Persona 3 → START=500, END=750
#   Persona 4 → START=750, END=None

START = 0       # primera fila (incluida)
END   = 4600    # última fila (excluida); None = hasta el final

df_chunk = enrich_with_years(
    input_csv="outputs/songs_clean_by_genre_exploded_transformer.csv",
    output_csv=None,           # genera automáticamente: years_0_500.csv, etc.
    overwrite_existing=False,
    start=START,
    end=END,
)

print(f"\nResumen del tramo {START}–{END or 'end'}:")
print(f"  Canciones procesadas: {len(df_chunk)}")
print(f"  Con año:              {df_chunk['year'].notna().sum()} ({df_chunk['year'].notna().mean():.0%})")
print(f"  Sin año:              {df_chunk['year'].isna().sum()}")


2026-05-09 10:54:22,573 [INFO] Rango: filas 0–4599 (4600 canciones) | Pendientes: 4600 | Salida: years_0_4600.csv
2026-05-09 10:54:22,573 [INFO] [1/4600] moonspell — Heartshaped Abyss
2026-05-09 10:54:22,574 [INFO] Probando: 'Heartshaped Abyss (Moonspell song)'
2026-05-09 10:54:23,871 [INFO] Request URL: https://en.wikipedia.org/w/api.php?format=json&redirects=1&action=query&prop=info&titles=Heartshaped Abyss (Moonspell song)&inprop=protection|talkid|watched|watchers|visitingwatchers|notificationtimestamp|subjectid|url|readable|preload|displaytitle|varianttitles
2026-05-09 10:54:24,066 [INFO] HTTP Request: GET https://en.wikipedia.org/w/api.php?format=json&redirects=1&action=query&prop=info&titles=Heartshaped+Abyss+%28Moonspell+song%29&inprop=protection%7Ctalkid%7Cwatched%7Cwatchers%7Cvisitingwatchers%7Cnotificationtimestamp%7Csubjectid%7Curl%7Creadable%7Cpreload%7Cdisplaytitle%7Cvarianttitles "HTTP/1.1 200 OK"
2026-05-09 10:54:24,067 [INFO] Probando: 'Heartshaped Abyss (Moonspell)'
20

KeyboardInterrupt: 

## Combinar tramos (ejecutar solo cuando todos hayan terminado)
Recoge los CSV parciales de cada uno y los fusiona en el dataset completo.

In [ ]:
# Pon aquí los nombres de los CSV que ha generado cada persona
df_final = merge_year_chunks(
    "years_0_4600.csv",
    "years_4601_end.csv",
    
    base_csv="music_dataset.csv",
    output_csv="music_dataset_with_years.csv",
)

print(f"\nDataset final:")
print(f"  Total canciones:  {len(df_final)}")
print(f"  Con año:          {df_final['year'].notna().sum()} ({df_final['year'].notna().mean():.0%})")
print(f"\nDistribución por década:")
print(df_final['year'].dropna().astype(int).floordiv(10).mul(10).value_counts().sort_index().to_string())
